# Stage 1-2 FITS-PSF flux and consistency evaluation

This notebook checks whether a bright HR reconstruction is physically/logically acceptable. The decisive comparison is not raw 128x128 SR versus raw 64x64 LR. It is the **re-degraded SR image** after fixed-SIS forward lensing, the same FITS PSF, and downsampling.

The repository currently min-max normalizes every input independently, so values are normalized intensities rather than calibrated physical flux. For grids with different pixel sizes, integrated flux is computed as `sum(intensity * pixel_area)`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm
import torch

import data
from differentiable_lensing import DifferentiableLensing
from psf import build_psf_kernel
from sisr import SISR
from evaluate_stage1_2_flux_utils import (
    image_metrics, load_sr_weights, psf_diagnostics, radial_profile,
    run_stage1_2_sample, weighted_centroid,
)

torch.set_grad_enabled(False)
pd.set_option('display.max_columns', 100)

## 1. Configuration
Edit the paths and FITS metadata below before running.

In [ ]:
WEIGHTS_PATH = Path('hsc_fits_psf_weights.pt')
PSF_FITS_PATH = Path('path/to/hsc_psf.fits')

VAL_DIR = 'val/'
CLASS_NAME = 'no_sub'       # no_sub, axion, or cdm
NUM_SAMPLES_IN_CLASS = 2000
SAMPLE_INDICES = [0, 1, 2]

LR_RESOLUTION = 0.168       # arcsec / LR pixel; set to your actual dataset value
IMAGE_SHAPE = 64
MAGNIFICATION = 2
N_MAG = 1
RESIDUAL_DEPTH = 3
IN_CHANNELS = 2
LATENT_CHANNELS = 64
DOWNSAMPLE_MODE = 'area'

PSF_HDU = 0                 # change after inspecting the FITS file
PSF_EXTNAME = None          # e.g. 'PSF'; overrides PSF_HDU when not None
PSF_NATIVE_PIXSCALE = 0.168 # arcsec / PSF pixel
PSF_CROP_SIZE = 65          # odd center crop; set None to keep all pixels

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
effective_magnification = MAGNIFICATION ** N_MAG
HR_RESOLUTION = LR_RESOLUTION / effective_magnification
TARGET_SHAPE = IMAGE_SHAPE * effective_magnification
print('device:', device)
print('LR pixel scale:', LR_RESOLUTION, 'arcsec/pixel')
print('HR pixel scale:', HR_RESOLUTION, 'arcsec/pixel')

## 2. Load the fixed-SIS Stage 1-2 pipeline and FITS PSF

In [ ]:
dataset = data.LensingDataset(VAL_DIR, [CLASS_NAME], NUM_SAMPLES_IN_CLASS)
model = SISR(
    magnification=MAGNIFICATION, n_mag=N_MAG, residual_depth=RESIDUAL_DEPTH,
    in_channels=IN_CHANNELS, latent_channel_count=LATENT_CHANNELS,
).to(device).eval()
load_sr_weights(model, str(WEIGHTS_PATH), device)

lensing_module = DifferentiableLensing(
    device=device, alpha=None, target_resolution=HR_RESOLUTION, target_shape=TARGET_SHAPE,
).to(device)
cross_grid_to_log = torch.load('scatter_to_log_128.pt', map_location=device).to(device)
cross_grid_forward_from_log = torch.load('forward_from_log_128.pt', map_location=device).to(device)
cross_grid_from_log = torch.load('scatter_from_log_128.pt', map_location=device).to(device)
cross_grid_backward = torch.load('sparse_grid_fracs_euclid_backward.pt', map_location=device).to(device)

psf_kernel = build_psf_kernel(
    psf_type='fits', fwhm_arcsec=0.16, pixscale_arcsec=HR_RESOLUTION,
    path=str(PSF_FITS_PATH), fits_hdu=PSF_HDU, fits_extname=PSF_EXTNAME,
    fits_crop_size=PSF_CROP_SIZE, source_pixscale_arcsec=PSF_NATIVE_PIXSCALE,
    device=device,
)
pd.DataFrame([psf_diagnostics(psf_kernel)])

### PSF checks
The PSF sum should be approximately 1. A centroid or peak offset larger than roughly one pixel suggests that the FITS stamp may need recentering before convolution. Also inspect whether the crop removes visible wings.

In [ ]:
k = psf_kernel.detach().cpu().numpy()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(k, origin='lower')
axes[0].set_title('Normalized FITS PSF')
axes[1].plot(k[k.shape[0] // 2])
axes[1].set_title('Central PSF row')
axes[1].set_xlabel('pixel')
plt.tight_layout()

## 3. Evaluate selected examples

In [ ]:
all_outputs = {}
rows = []
for index in SAMPLE_INDICES:
    lr = dataset[index].float().to(device)
    outputs = run_stage1_2_sample(
        lr, model, lensing_module, cross_grid_backward, cross_grid_to_log,
        cross_grid_forward_from_log, cross_grid_from_log, psf_kernel,
        effective_magnification, DOWNSAMPLE_MODE,
    )
    all_outputs[index] = outputs
    row = {'index': index, **image_metrics(outputs, LR_RESOLUTION, HR_RESOLUTION)}
    rows.append(row)
metrics_df = pd.DataFrame(rows)
metrics_df

### How to interpret the table
- `hr_to_lr_peak_ratio > 1` is not automatically wrong; a deconvolved HR image can have a higher peak.
- `redegraded_to_lr_peak_ratio` should be much closer to 1.
- `redegraded_to_lr_flux_ratio` should be close to 1 when the operator and normalization are consistent.
- Low `lr_redegradation_nmse`, high PSNR, and correlation near 1 are stronger evidence than visual brightness alone.
- Because inputs are min-max normalized, these are relative rather than calibrated flux tests.

In [ ]:
def show_case(index):
    o = all_outputs[index]
    names = ['lr', 'source_lr', 'source_hr', 'image_hr', 'image_hr_psf', 'lr_redegraded', 'residual_lr']
    titles = ['LR observed', 'LR source reconstruction', 'SR source', 'Raw HR lensed',
              'HR after FITS PSF', 'Re-degraded SR on LR grid', 'LR residual: predicted - observed']
    arrays = [o[n][0, 0].detach().cpu().numpy() for n in names]
    vmax_lr = max(arrays[0].max(), arrays[5].max())
    vmax_hr = max(arrays[3].max(), arrays[4].max())
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.ravel()
    for ax, arr, title in zip(axes, arrays, titles):
        if 'residual' in title.lower():
            lim = max(abs(arr.min()), abs(arr.max()), 1e-8)
            im = ax.imshow(arr, origin='lower', cmap='coolwarm', vmin=-lim, vmax=lim)
        elif title in ['LR observed', 'Re-degraded SR on LR grid']:
            im = ax.imshow(arr, origin='lower', cmap='gray', vmin=0, vmax=vmax_lr)
        elif title in ['Raw HR lensed', 'HR after FITS PSF']:
            im = ax.imshow(arr, origin='lower', cmap='gray', vmin=0, vmax=vmax_hr)
        else:
            im = ax.imshow(arr, origin='lower', cmap='gray')
        ax.set_title(title)
        fig.colorbar(im, ax=ax, fraction=0.046)
    axes[-1].axis('off')
    fig.suptitle(f'{CLASS_NAME} sample {index}')
    plt.tight_layout()

for idx in SAMPLE_INDICES:
    show_case(idx)

## 4. Per-pixel intensity/flux distributions and residuals

In [ ]:
for index, o in all_outputs.items():
    lr = o['lr'][0, 0].cpu().numpy()
    re = o['lr_redegraded'][0, 0].cpu().numpy()
    hr = o['image_hr'][0, 0].cpu().numpy()
    residual = re - lr

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].hist(lr.ravel(), bins=80, alpha=0.6, label='LR observed')
    axes[0].hist(re.ravel(), bins=80, alpha=0.6, label='LR re-degraded')
    axes[0].set_yscale('log'); axes[0].legend(); axes[0].set_title('Pixel-value distribution')
    axes[1].scatter(lr.ravel(), re.ravel(), s=3, alpha=0.25)
    lo, hi = min(lr.min(), re.min()), max(lr.max(), re.max())
    axes[1].plot([lo, hi], [lo, hi], '--')
    axes[1].set_xlabel('LR observed'); axes[1].set_ylabel('LR re-degraded'); axes[1].set_title('Per-pixel agreement')
    axes[2].hist(residual.ravel(), bins=100)
    axes[2].axvline(0, linestyle='--'); axes[2].set_title('Residual distribution')
    fig.suptitle(f'Sample {index}')
    plt.tight_layout()

## 5. Radial profiles and tangential/ring brightness

In [ ]:
for index, o in all_outputs.items():
    lr = o['lr'][0, 0].cpu().numpy()
    re = o['lr_redegraded'][0, 0].cpu().numpy()
    hr = o['image_hr'][0, 0].cpu().numpy()
    center_lr = weighted_centroid(lr)
    center_hr = (center_lr[0] * effective_magnification, center_lr[1] * effective_magnification)
    r_lr, p_lr = radial_profile(lr, center_lr)
    r_re, p_re = radial_profile(re, center_lr)
    r_hr, p_hr = radial_profile(hr, center_hr)
    plt.figure(figsize=(7, 4))
    plt.plot(r_lr * LR_RESOLUTION, p_lr, label='LR observed')
    plt.plot(r_re * LR_RESOLUTION, p_re, label='LR re-degraded')
    plt.plot(r_hr * HR_RESOLUTION, p_hr, label='raw HR lensed', alpha=0.8)
    plt.xlabel('radius [arcsec]'); plt.ylabel('mean intensity per annulus')
    plt.title(f'Radial brightness profile: sample {index}')
    plt.legend(); plt.grid(alpha=0.2); plt.show()

## 6. Aggregate validation check

In [ ]:
# Increase these lists after the notebook works on a few examples.
classes = ['no_sub', 'axion', 'cdm']
aggregate_indices = list(range(20))
aggregate_rows = []
for cls in classes:
    ds = data.LensingDataset(VAL_DIR, [cls], NUM_SAMPLES_IN_CLASS)
    for index in aggregate_indices:
        outputs = run_stage1_2_sample(
            ds[index].float().to(device), model, lensing_module, cross_grid_backward,
            cross_grid_to_log, cross_grid_forward_from_log, cross_grid_from_log,
            psf_kernel, effective_magnification, DOWNSAMPLE_MODE,
        )
        aggregate_rows.append({'class': cls, 'index': index, **image_metrics(outputs, LR_RESOLUTION, HR_RESOLUTION)})
aggregate_df = pd.DataFrame(aggregate_rows)
aggregate_df.groupby('class').agg(['mean', 'std', 'median'])[[
    'hr_to_lr_peak_ratio', 'redegraded_to_lr_peak_ratio', 'redegraded_to_lr_flux_ratio',
    'lr_redegradation_nmse', 'lr_redegradation_psnr', 'lr_redegradation_correlation',
]]

## Decision guide

A bright raw SR tangent is probably acceptable when:
1. the FITS PSF is normalized and centered;
2. the re-degraded SR closely matches the LR image spatially and photometrically;
3. the area-weighted re-degraded/LR flux ratio is near 1 across many samples;
4. bright residuals are not systematically concentrated at tangential arcs; and
5. results are stable across checkpoints and reasonable PSF crops.

It is an issue when the re-degraded image is also too bright, residuals show coherent positive arc-shaped structures, the PSF is off-center/mis-scaled, or the peak/flux bias is systematic across the validation set.

Because `data.py` min-max normalizes each object, this notebook cannot establish physical flux conservation in electrons, Jy, or counts. For scientific photometry, preserve calibrated image units and carry background/variance information instead of per-image min-max normalization.